# 06 · Evaluation, error analysis and the comparison

**ITRI626 · Colile Sibanda**

Objectives **O6**, **O7** and **O8**. Every number quoted in the report
and the presentation comes from this notebook's outputs, and every one of
them appears in `outputs/metrics/comparison.csv`.

This is the only notebook that touches the held-out split.


In [ ]:
# Every notebook opens with the same four lines: path, config, seed, paths.
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT))   # run_status.py and run_compare.py live at the root

from docclf.config.loader import load_config
from docclf.config.paths import resolve_paths
from docclf.config.seed import set_global_seed
from docclf.utils.env import detect_environment

CONFIG = ROOT / "configs" / "base.yaml"
ENV = detect_environment()
cfg = load_config(CONFIG, env=ENV)
paths = resolve_paths(cfg, ENV).ensure()
set_global_seed(cfg.seed)

f"Config: {cfg.name} | Environment: {ENV} | Seed: {cfg.seed}"

In [ ]:
# Every model that has been trained and evaluated. Each writes into its own run
# directory, so the comparison can combine runs finished on different machines.
# A model that is not evaluated yet is dropped with a note, which makes this
# notebook usable before the last run has landed.
from run_status import MODELS as RUN_SPECS

MODELS = ['baseline_cnn', 'baseline_cnn_v2', 'resnet50', 'vit_b16']
RUN_DIR = {name: ROOT / RUN_SPECS[name]['dir'] for name in MODELS}

available = [
    name for name in MODELS
    if (RUN_DIR[name] / 'metrics' / f'metrics_{name}.csv').is_file()
]
missing = [name for name in MODELS if name not in available]
if missing:
    print(f'Not evaluated yet, so left out: {", ".join(missing)}')
MODELS = available
MODELS


## Load the committed split and re-verify it


In [ ]:
from docclf.splits.io import load_splits
from docclf.splits.leakage import assert_no_group_overlap
from docclf.data.loaders import build_dataloaders
from docclf.utils.env import select_device

# `root=` resolves the split files' relative image paths against this machine,
# which is what lets the same committed split files work everywhere.
splits = load_splits(paths.splits, root=paths.root)
assert_no_group_overlap(splits, cfg.split.group_column)
loaders = build_dataloaders(splits, cfg)
device = select_device(cfg)
print(f'Held-out pages: {len(loaders["test"].dataset)}')
print(f'Device: {device}')


## Predict with each trained model

Each model is restored from its best checkpoint and run over the same held-out loader.


In [ ]:
# Each model's held-out predictions were written by `run_evaluate.py` when that
# model finished, possibly on another machine. They are read back here rather
# than recomputed: re-running inference for four models would take hours on CPU
# and would need every checkpoint present on this one machine.
#
# The saved table keeps the winning class's confidence rather than the full
# logit vector, so the metrics, confusion matrices and error analysis below are
# exact. Anything needing the full distribution would have to re-run
# `collect_predictions` against the checkpoints.
import pandas as pd

from docclf.evaluation.metrics import compute_metrics
from docclf.evaluation.efficiency import EfficiencySummary
from docclf.evaluation.predict import load_predictions

predictions, metrics, efficiency = {}, {}, {}
for name in MODELS:
    metrics_dir = RUN_DIR[name] / 'metrics'
    predictions[name] = load_predictions(
        metrics_dir / f'predictions_{name}.csv', cfg.classes
    )
    metrics[name] = compute_metrics(predictions[name], name)
    efficiency_path = metrics_dir / f'efficiency_{name}.csv'
    if efficiency_path.is_file():
        row = pd.read_csv(efficiency_path).iloc[0]
        efficiency[name] = EfficiencySummary(
            model_name=str(row['model_name']),
            device=str(row['device']),
            batch_size=int(row['batch_size']),
            repeats=int(row['repeats']),
            median_ms_per_image=float(row['median_ms_per_image']),
            p90_ms_per_image=float(row['p90_ms_per_image']),
            total_parameters=int(row['total_parameters']),
            size_mb=float(row['size_mb']),
        )
    print(f'{name}: accuracy {metrics[name].accuracy:.4f} '
          f'| macro-F1 {metrics[name].macro_f1:.4f}')


## The comparison table

All three models, one held-out split. `comparison.csv` is the single source of every number in the report.


In [ ]:
from docclf.evaluation.compare import build_comparison_table, save_comparison, recommend
from docclf.models.registry import model_family

families = {name: model_family(name) for name in MODELS}
comparison = build_comparison_table(metrics, efficiency, families)
save_comparison(comparison, paths.metrics)
comparison

## Per-class performance


In [ ]:
import pandas as pd

per_class = pd.concat(
    [metrics[name].per_class.assign(model=name) for name in MODELS], ignore_index=True
)
pivot = per_class.pivot(index='class_name', columns='model', values='f1').reindex(cfg.classes)
ax = pivot.plot.bar(figsize=(11, 4))
ax.set_ylabel('F1')
ax.set_ylim(0, 1)
ax.set_title('Per-class F1 on the held-out split')
pivot.round(4)

## Confusion matrices

Rows are true classes, columns predicted — stated on the figure itself.


In [ ]:
from docclf.evaluation.confusion import confusion_frame, most_confused_pairs, plot_confusion
from IPython.display import Image as Show, display

for name in MODELS:
    path = plot_confusion(predictions[name], paths.figures / f'confusion_{name}.png', name)
    plot_confusion(predictions[name], paths.figures / f'confusion_normalised_{name}.png', name, normalise=True)
    display(Show(str(path)))

## Error analysis (objective O7)

Which pairs are confused, in which direction, and what the pages look like.


In [ ]:
pairs = pd.concat(
    [most_confused_pairs(predictions[name], top_n=5).assign(model=name) for name in MODELS],
    ignore_index=True,
)
pairs.to_csv(paths.metrics / 'confused_pairs.csv', index=False)
pairs

In [ ]:
from docclf.evaluation.errors import plot_error_examples, sample_errors

best_model = comparison.iloc[0]['model']
path = plot_error_examples(
    predictions[best_model], paths.figures / f'errors_{best_model}.png', cfg.eval.error_samples, best_model
)
display(Show(str(path)))
sample_errors(predictions[best_model], cfg.eval.error_samples)

## Efficiency and the recommendation (objective O8)

Accuracy is only half the argument. State the latency, the size, the
device they were measured on, and the conditions under which the
recommendation would change.


In [ ]:
efficiency_table = pd.DataFrame([efficiency[name].as_row() for name in MODELS])
efficiency_table.to_csv(paths.metrics / 'efficiency.csv', index=False)
efficiency_table

In [ ]:
print(recommend(comparison, cfg.eval.review_threshold))
print()
print('The recommendation changes if: the intake queue needs a lower per-page')
print('latency than the leading model provides; or the deployment target has')
print('no GPU, which moves the argument towards the smaller model.')